# Notebook 7 – Binning & Discretization

## Learning Objectives
Understand binning; apply equal-width, equal-frequency, quantile, and domain-based binning; convert continuous values into categories; create age, income, and risk groups; explain advantages and limitations.

## 1. What is Binning?
Binning (discretization) converts a continuous numeric variable into intervals or categories.

**Example:** Group exact customer ages into Young, Adult, and Senior categories.

## 2. Why Binning?
Binning can simplify numeric values, improve interpretability, represent threshold-based relationships, and reduce sensitivity to small fluctuations. It also loses detail because different values in one interval become the same category.

## 3. Sample Dataset

In [1]:
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "customer_id":range(1,13),
    "age":[18,22,25,29,33,37,41,46,52,58,64,72],
    "annual_income":[180000,220000,280000,320000,400000,450000,520000,610000,700000,850000,1000000,1300000],
    "risk_score":[12,18,22,27,31,38,42,49,55,63,74,88]
})
customers

,customer_id,age,annual_income,risk_score
0,1,18,180000,12
1,2,22,220000,18
2,3,25,280000,22
3,4,29,320000,27
4,5,33,400000,31
5,6,37,450000,38
6,7,41,520000,42
7,8,46,610000,49
8,9,52,700000,55
9,10,58,850000,63


**Output Explanation:** Age and income are continuous values. Risk score is numeric and will be grouped for illustration.

## 4. Equal-Width Binning
Equal-width binning divides the numeric range into intervals of equal width.

In [2]:
customers["age_equal_width"] = pd.cut(
    customers["age"], bins=4,
    labels=["Bin 1","Bin 2","Bin 3","Bin 4"],
    include_lowest=True
)
customers[["age","age_equal_width"]]

,age,age_equal_width
0,18,Bin 1
1,22,Bin 1
2,25,Bin 1
3,29,Bin 1
4,33,Bin 2
5,37,Bin 2
6,41,Bin 2
7,46,Bin 3
8,52,Bin 3
9,58,Bin 3


**Output Explanation:** Each interval covers approximately the same age range, but the number of customers in each interval may differ.

## 5. Equal-Frequency Binning
Equal-frequency binning aims to place a similar number of observations in each bin. Quantile binning is a common implementation.

In [3]:
customers["age_equal_frequency"] = pd.qcut(
    customers["age"], q=4,
    labels=["Q1","Q2","Q3","Q4"]
)
customers[["age","age_equal_frequency"]]

,age,age_equal_frequency
0,18,Q1
1,22,Q1
2,25,Q1
3,29,Q2
4,33,Q2
5,37,Q2
6,41,Q3
7,46,Q3
8,52,Q3
9,58,Q4


**Output Explanation:** Each group contains approximately the same number of records. The numeric widths of the intervals may differ.

## 6. Quantile Binning
Quantile binning uses distribution-based cut points, such as quartiles or deciles.

In [4]:
customers["income_quartile"] = pd.qcut(
    customers["annual_income"], q=4,
    labels=["Low","Lower-Middle","Upper-Middle","High"]
)
customers[["annual_income","income_quartile"]]

,annual_income,income_quartile
0,180000,Low
1,220000,Low
2,280000,Low
3,320000,Lower-Middle
4,400000,Lower-Middle
5,450000,Lower-Middle
6,520000,Upper-Middle
7,610000,Upper-Middle
8,700000,Upper-Middle
9,850000,High


**Output Explanation:** Groups are based on this sample's income distribution. In predictive modelling, learn boundaries from training data only.

## 7. Domain-Based Binning
Domain-based binning uses fixed thresholds chosen from business knowledge or policy.

In [5]:
age_bins = [0,25,40,60,np.inf]
age_labels = ["Young","Adult","Middle-aged","Senior"]
customers["age_group"] = pd.cut(
    customers["age"], bins=age_bins, labels=age_labels, right=True
)
customers[["age","age_group"]]

,age,age_group
0,18,Young
1,22,Young
2,25,Young
3,29,Adult
4,33,Adult
5,37,Adult
6,41,Middle-aged
7,46,Middle-aged
8,52,Middle-aged
9,58,Middle-aged


**Output Explanation:** These illustrative fixed ranges are easy to explain but are not universal demographic definitions.

## 8. Continuous to Categorical
Discretization converts numeric values into categories.

In [6]:
risk_bins = [-np.inf,30,60,np.inf]
risk_labels = ["Low Risk","Medium Risk","High Risk"]
customers["risk_category"] = pd.cut(
    customers["risk_score"], bins=risk_bins, labels=risk_labels
)
customers[["risk_score","risk_category"]]

,risk_score,risk_category
0,12,Low Risk
1,18,Low Risk
2,22,Low Risk
3,27,Low Risk
4,31,Medium Risk
5,38,Medium Risk
6,42,Medium Risk
7,49,Medium Risk
8,55,Medium Risk
9,63,High Risk


**Output Explanation:** Numeric risk scores are represented by groups. Real banking or insurance thresholds must be defined and validated by domain experts.

## 9. Duplicate Quantile Boundaries
Repeated values can produce duplicate quantile boundaries. `duplicates="drop"` removes duplicate boundaries, possibly creating fewer bins than requested.

In [7]:
repeated = pd.Series([10,10,10,20,20,30,30,30,40,40])
result = pd.qcut(repeated, q=4, duplicates="drop")
print(result)
print("Actual number of bins:", result.cat.categories.size)

0    (9.999, 12.5]
1    (9.999, 12.5]
2    (9.999, 12.5]
3     (12.5, 25.0]
4     (12.5, 25.0]
5     (25.0, 30.0]
6     (25.0, 30.0]
7     (25.0, 30.0]
8     (30.0, 40.0]
9     (30.0, 40.0]
dtype: category
Categories (4, interval[float64, right]): [(9.999, 12.5] < (12.5, 25.0] < (25.0, 30.0] < (30.0, 40.0]]
Actual number of bins: 4


**Output Explanation:** Always inspect the actual number of bins, especially when the data contains many repeated values.